## 1. Introduction
This notebook demontrates the core functionalities of **NumCompute**, a lightweight and fully vectorized machine learning framework built from scratch using Numpy.

In [1]:
import numpy as np 
from numcompute.io import load_csv

#Loading a dataset using out CSV Loader 
print("---- 1. Testing CSV Loader ----")
try:
    data = load_csv("data_csv", delimiter=",", skiprows=0)
    print("Data loaded successfully from the .csv file! Shape:", data.shape)
except Exception as e:
    #Returning fallback dummy dataset if data.csv is not found
    print("Creating fallback dataset with missing values...")
    data = np.array([
        [1.0, np.nan, 2.0],
        [3.0, 4.0, np.nan],
        [5.0, 6.0, 7.0],
        [1.0, 4.0, 2.0]
        ])
    print("Synthetic data shape:", data.shape)



---- 1. Testing CSV Loader ----
Creating fallback dataset with missing values...
Synthetic data shape: (4, 3)


## 2. Data Preprocessing
We utilize our vectorized transformers (`SimpleImputer`, `StandardScaler`, `MinMaxScaler`, and `OneHotEncoder`) to clean and scale the dataset without Python loops.

In [2]:
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler

print("\n---- 2. Testing Preprocessing Transformers ----")

#Handle missing values using mean imputation
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(data)
print("After Imputation:\n", X_imputed)

#Scale features to zero mean and unit variance using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print("\nAfter StandardScaler:\n", X_scaled)



---- 2. Testing Preprocessing Transformers ----
After Imputation:
 [[1.         4.66666667 2.        ]
 [3.         4.         3.66666667]
 [5.         6.         7.        ]
 [1.         4.         2.        ]]

After StandardScaler:
 [[-0.90453403  0.         -0.81649658]
 [ 0.30151134 -0.81649658  0.        ]
 [ 1.50755672  1.63299316  1.63299316]
 [-0.90453403 -0.81649658 -0.81649658]]


## 3. Chaining Steps with Pipeline
We can seamlessly chain multiple preprocessing steps together using the `Pipeline` class, following standard scikit-learn workflows.

In [3]:
from numcompute.pipeline import Pipeline

print("\n---- 2. Testing Pipeline ----")

#Define a sequential pipeline
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', MinMaxScaler(feature_range=(0.0, 1.0)))
])

#Fit and transform
X_pipeline_result = pipe.fit_transform(data)
print("Pipeline final output:\n", X_pipeline_result)

print("\nYour preprocessing and pipeline sections are successfully set up! You're ready for the next module.")



---- 2. Testing Pipeline ----
Pipeline final output:
 [[0.         0.33333333 0.        ]
 [0.5        0.         0.33333333]
 [1.         1.         1.        ]
 [0.         0.         0.        ]]

Your preprocessing and pipeline sections are successfully set up! You're ready for the next module.


## 4. Sorting, Searching and Ranking
`sort_search` provides top-k selection (`np.argpartition`), quickselect and binary search; `rank` provides tie-aware ranking and percentiles. NaN is treated as missing: it is never selected before a real value, stays unranked, and is ignored by `percentile`.

In [4]:
from numcompute.sort_search import topk, quickselect, binary_search
from numcompute.rank import rank, percentile

print("\n---- 4. Testing Sorting, Searching and Ranking ----")

#Scores with a tie (95) and a missing value (NaN)
scores = np.array([72, 95, 88, 95, 60, np.nan, 81])

#Top-k: the two tied 95s keep their original order (index 1, then 3)
values, indices = topk(scores, 3)
print("Top-3 values:", values, "| indices:", indices)

#Quickselect: k-th smallest value (0-based) without sorting the whole array
print("3rd smallest (k=2):", quickselect(np.array([72, 95, 88, 95, 60, 81]), 2, seed=0))

#Binary search: insertion index and whether the value exists
index, found = binary_search(np.array([60, 72, 81, 88, 95]), [81, 90])
print("Binary search for [81, 90] -> index:", index, "| found:", found)

#Ranking with three tie-handling methods (NaN stays unranked)
for method in ["average", "dense", "ordinal"]:
    print(f"Rank ({method}):", rank(scores, method=method))

#Percentiles ignore NaN; four interpolation methods are supported
print("Percentiles (25, 50, 75):", percentile(scores, [25, 50, 75]))
print("Median (linear vs lower):", percentile(scores, 50), "vs", percentile(scores, 50, interpolation="lower"))

#Ranking the first column of the preprocessed dataset from section 2
print("\nRank of column 0 in X_imputed:", rank(X_imputed[:, 0], method="average"))


---- 4. Testing Sorting, Searching and Ranking ----
Top-3 values: [95. 95. 88.] | indices: [1 3 2]
3rd smallest (k=2): 81
Binary search for [81, 90] -> index: [2 4] | found: [ True False]
Rank (average): [2.  5.5 4.  5.5 1.  nan 3. ]
Rank (dense): [ 2.  5.  4.  5.  1. nan  3.]
Rank (ordinal): [ 2.  5.  4.  6.  1. nan  3.]
Percentiles (25, 50, 75): [74.25 84.5  93.25]
Median (linear vs lower): 84.5 vs 81.0

Rank of column 0 in X_imputed: [1.5 3.  4.  1.5]
